# 🔗 Graph RAG デモ — スキーマ自動取得版

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
[![Python](https://img.shields.io/badge/Python-3.10%2B-blue?logo=python&logoColor=white)](https://www.python.org/)
[![Neo4j](https://img.shields.io/badge/Neo4j-5.18-008CC1?logo=neo4j&logoColor=white)](https://neo4j.com/)
[![Ollama](https://img.shields.io/badge/Ollama-llama3-black)](https://ollama.com/)

## このノートブックの特徴

| | スキーマ固定版 | **スキーマ自動取得版（本ノートブック）** |
|---|---|---|
| スキーマの指定 | プロンプトにハードコード | Neo4j から実行時に自動取得 |
| Cypher の生成 | LLM が直接生成（誤りが多い） | LLM は意図だけ、Cypher はコードで生成 |
| DB 変更時 | プロンプトの書き直しが必要 | **変更不要・自動対応** |
| ホップ数の保証 | LLM 次第でまとめられる | **ホップ数 = クエリ数を保証** |

## アーキテクチャ

```
ユーザークエリ
      │
      ▼
  Neo4j にスキーマを問い合わせ  ← CALL db.labels() 等
      │
      ▼
  Ollama（アプリ層）
  「何を・どのリレーションで・どの順にたどるか」だけ決める
  Cypher の文字列生成はしない
      │
      ▼
  コードが Cypher を自動生成
  1ステップ = 1リレーション = 1クエリ を保証
      │
   ┌──┴──┬──────┐
   ▼     ▼      ▼
 クエリ① ② ③  ← Neo4j（DB層）各クエリを個別に実行
   └──┬──┴──────┘
      ▼
  Ollama（アプリ層）結果を統合して最終回答
```

⚠️ **注意**: STEP 1〜2 は1セッション内で2回実行しないでください。

---
## STEP 1 — Neo4j インストール
> ⏱️ 約2〜3分

In [ ]:
%%bash
apt-get install -y openjdk-17-jdk-headless zstd -qq
wget -q https://dist.neo4j.org/neo4j-community-5.18.0-unix.tar.gz
tar -xzf neo4j-community-5.18.0-unix.tar.gz
mv neo4j-community-5.18.0 /opt/neo4j
echo "✅ Neo4j インストール完了"

---
## STEP 2 — Neo4j 設定と起動
⚠️ このセルは **1セッションにつき1回だけ** 実行してください

In [ ]:
%%bash
conf=/opt/neo4j/conf/neo4j.conf
grep -q 'server.bolt.listen_address'    $conf || echo 'server.bolt.listen_address=0.0.0.0:7687'    >> $conf
grep -q 'server.http.listen_address'    $conf || echo 'server.http.listen_address=0.0.0.0:7474'    >> $conf
grep -q 'server.default_listen_address' $conf || echo 'server.default_listen_address=0.0.0.0'      >> $conf
/opt/neo4j/bin/neo4j-admin server validate-config
echo "✅ 設定検証完了"

In [ ]:
import subprocess, time, os, socket

env = os.environ.copy()
env['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
subprocess.Popen(
    ['/opt/neo4j/bin/neo4j', 'console'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=env
)
print('Neo4j 起動中...')
for i in range(9):
    time.sleep(10)
    try:
        s = socket.create_connection(('localhost', 7687), timeout=3)
        s.close()
        print(f'✅ Neo4j 起動完了（{(i+1)*10}秒）')
        break
    except:
        print(f'  待機中... {(i+1)*10}秒経過')
else:
    print('❌ 起動タイムアウト')
    r = subprocess.run(['tail', '-20', '/opt/neo4j/logs/neo4j.log'], capture_output=True, text=True)
    print(r.stdout)

---
## STEP 3 — Ollama インストールとモデルダウンロード
> ⏱️ llama3 は約 4.7GB。`phi3`（軽量版・約2.3GB）に変更も可能

In [ ]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh
echo "✅ Ollama インストール完了"
ollama --version

In [ ]:
import subprocess, time, os

env = os.environ.copy()
env['OLLAMA_HOST'] = '0.0.0.0'
subprocess.Popen(['ollama', 'serve'], env=env,
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# ── モデル選択 ──────────────────────────────────────
MODEL_NAME = 'llama3'   # 軽量版にする場合は 'phi3' に変更
# ────────────────────────────────────────────────────

print(f"モデル '{MODEL_NAME}' をダウンロード中...")
subprocess.run(['ollama', 'pull', MODEL_NAME])
print(f"✅ モデル '{MODEL_NAME}' 準備完了")

---
## STEP 4 — ライブラリインストールと日本語フォント設定

In [ ]:
!pip install neo4j requests networkx matplotlib -q
print('✅ ライブラリインストール完了')

In [ ]:
import os, subprocess
subprocess.run(['apt-get', 'install', '-y', 'fonts-ipafont'], capture_output=True)
os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib'
os.makedirs('/tmp/matplotlib', exist_ok=True)

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
fm.fontManager.addfont('/usr/share/fonts/opentype/ipafont-gothic/ipagp.ttf')
plt.rcParams['font.family'] = 'IPAPGothic'

fig, ax = plt.subplots(figsize=(4, 1.5))
ax.text(0.5, 0.5, 'グラフRAGデモ：日本語表示OK', ha='center', va='center', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()
print('✅ 日本語フォント設定完了（IPAPGothic）')

---
## STEP 5 — Neo4j 接続とサンプルデータ投入

In [ ]:
from neo4j import GraphDatabase
from neo4j.graph import Node, Relationship

NEO4J_URI      = 'bolt://localhost:7687'
NEO4J_USER     = 'neo4j'
NEO4J_PASSWORD = 'demo1234'

# 初回のみ：初期パスワードを変更
try:
    driver_init = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, 'neo4j'))
    with driver_init.session(database='system') as session:
        session.run(f"ALTER CURRENT USER SET PASSWORD FROM 'neo4j' TO '{NEO4J_PASSWORD}'")
    driver_init.close()
    print('✅ パスワード変更完了')
except Exception as e:
    print(f'パスワード変更スキップ（既に変更済み）: {e}')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def serialize_value(val):
    """Neo4j の Node/Relationship オブジェクトを辞書に変換する"""
    if isinstance(val, Node):
        return {'labels': list(val.labels), 'properties': dict(val)}
    elif isinstance(val, Relationship):
        return {'type': val.type, 'properties': dict(val)}
    elif isinstance(val, list):
        return [serialize_value(v) for v in val]
    return val

def run_cypher(query: str, params: dict = {}):
    with driver.session() as session:
        results = []
        for r in session.run(query, params):
            results.append({k: serialize_value(v) for k, v in dict(r).items()})
        return results

run_cypher('RETURN 1 AS test')
print('✅ Neo4j 接続成功')

In [ ]:
run_cypher('MATCH (n) DETACH DELETE n')

setup_queries = [
    "CREATE (:Person {name: '田中太郎', role: 'engineer'})",
    "CREATE (:Person {name: '佐藤花子', role: 'manager'})",
    "CREATE (:Person {name: '鈴木一郎', role: 'engineer'})",
    "CREATE (:Person {name: '山田次郎', role: 'director'})",
    "CREATE (:Person {name: '高橋美咲', role: 'engineer'})",
    "CREATE (:Org {name: '開発部'})",
    "CREATE (:Org {name: '営業部'})",
    "CREATE (:Company {name: '株式会社ABC'})",
    "CREATE (:Project {name: 'プロジェクトX', status: 'active'})",
    "CREATE (:Project {name: 'プロジェクトY', status: 'completed'})",
    "MATCH (p:Person {name:'田中太郎'}),(o:Org {name:'開発部'})    CREATE (p)-[:BELONGS_TO]->(o)",
    "MATCH (p:Person {name:'鈴木一郎'}),(o:Org {name:'開発部'})    CREATE (p)-[:BELONGS_TO]->(o)",
    "MATCH (p:Person {name:'高橋美咲'}),(o:Org {name:'開発部'})    CREATE (p)-[:BELONGS_TO]->(o)",
    "MATCH (p:Person {name:'佐藤花子'}),(o:Org {name:'営業部'})    CREATE (p)-[:BELONGS_TO]->(o)",
    "MATCH (p:Person {name:'山田次郎'}),(o:Org {name:'開発部'})    CREATE (p)-[:MANAGES]->(o)",
    "MATCH (p:Person {name:'山田次郎'}),(o:Org {name:'営業部'})    CREATE (p)-[:MANAGES]->(o)",
    "MATCH (o:Org {name:'開発部'}),(c:Company {name:'株式会社ABC'})  CREATE (o)-[:PART_OF]->(c)",
    "MATCH (o:Org {name:'営業部'}),(c:Company {name:'株式会社ABC'})  CREATE (o)-[:PART_OF]->(c)",
    "MATCH (p:Person {name:'山田次郎'}),(c:Company {name:'株式会社ABC'}) CREATE (p)-[:REPRESENTS]->(c)",
    "MATCH (p:Person {name:'田中太郎'}),(j:Project {name:'プロジェクトX'}) CREATE (p)-[:WORKS_ON]->(j)",
    "MATCH (p:Person {name:'鈴木一郎'}),(j:Project {name:'プロジェクトX'}) CREATE (p)-[:WORKS_ON]->(j)",
    "MATCH (p:Person {name:'高橋美咲'}),(j:Project {name:'プロジェクトY'}) CREATE (p)-[:WORKS_ON]->(j)",
    "MATCH (p:Person {name:'佐藤花子'}),(j:Project {name:'プロジェクトY'}) CREATE (p)-[:WORKS_ON]->(j)",
]
for q in setup_queries:
    run_cypher(q)

nodes = run_cypher('MATCH (n) RETURN count(n) AS total')[0]['total']
edges = run_cypher('MATCH ()-[r]->() RETURN count(r) AS total')[0]['total']
print(f'✅ データ投入完了 — ノード: {nodes}件 / リレーション: {edges}件')

---
## STEP 6 — グラフの可視化

In [ ]:
import os; os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib'
import matplotlib.font_manager as fm, matplotlib.pyplot as plt
fm.fontManager.addfont('/usr/share/fonts/opentype/ipafont-gothic/ipagp.ttf')
plt.rcParams['font.family'] = 'IPAPGothic'

import networkx as nx
from matplotlib.patches import Patch

G_vis = nx.DiGraph()
for row in run_cypher('MATCH (n) RETURN labels(n)[0] AS label, n.name AS name'):
    G_vis.add_node(row['name'], label=row['label'])
for row in run_cypher('MATCH (a)-[r]->(b) RETURN a.name AS src, type(r) AS rel, b.name AS dst'):
    G_vis.add_edge(row['src'], row['dst'], rel=row['rel'])

color_map = {'Person': '#7F77DD', 'Org': '#1D9E75', 'Company': '#EF9F27', 'Project': '#D85A30'}
node_colors = [color_map.get(G_vis.nodes[n].get('label', ''), '#aaa') for n in G_vis.nodes]
pos = nx.spring_layout(G_vis, seed=42, k=3.0)
plt.figure(figsize=(13, 8))
nx.draw_networkx_nodes(G_vis, pos, node_color=node_colors, node_size=2000, alpha=0.9)
nx.draw_networkx_labels(G_vis, pos, font_size=9, font_color='white',
                        font_weight='bold', font_family='IPAPGothic')
nx.draw_networkx_edges(G_vis, pos, arrows=True, arrowsize=20,
                       edge_color='#888', connectionstyle='arc3,rad=0.1', width=1.5)
nx.draw_networkx_edge_labels(G_vis, pos, nx.get_edge_attributes(G_vis, 'rel'),
                              font_size=7, font_color='#444', font_family='IPAPGothic')
plt.legend(handles=[Patch(color=v, label=k) for k, v in color_map.items()], loc='upper left')
plt.title('Neo4j グラフDB の構造', fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.show()

---
## STEP 7 — スキーマ自動取得関数

Neo4j の組み込みプロシージャでスキーマを取得します。
これをプロンプトに動的に埋め込むことで、DBのスキーマが変わっても自動対応できます。

| プロシージャ | 取得できる情報 |
|---|---|
| `CALL db.labels()` | ノードラベル一覧 |
| `CALL db.relationshipTypes()` | リレーションシップ種別一覧 |
| `CALL db.schema.nodeTypeProperties()` | 各ノードのプロパティ一覧 |
| `CALL db.schema.relTypeProperties()` | 各リレーションのプロパティ一覧 |

In [ ]:
def get_schema() -> str:
    """Neo4j からスキーマ情報を自動取得してテキスト形式で返す"""
    labels = run_cypher('CALL db.labels() YIELD label RETURN label ORDER BY label')
    rels   = run_cypher('CALL db.relationshipTypes() YIELD relationshipType RETURN relationshipType ORDER BY relationshipType')
    node_props = run_cypher("""
        CALL db.schema.nodeTypeProperties()
        YIELD nodeLabels, propertyName, propertyTypes
        RETURN nodeLabels, propertyName, propertyTypes
        ORDER BY nodeLabels, propertyName
    """)
    rel_props = run_cypher("""
        CALL db.schema.relTypeProperties()
        YIELD relType, propertyName, propertyTypes
        RETURN relType, propertyName, propertyTypes
        ORDER BY relType, propertyName
    """)

    lines = ['=== Neo4j スキーマ（自動取得）===']
    lines.append('\n【ノードラベル】')
    for r in labels:
        lines.append(f'  - :{r["label"]}')
    lines.append('\n【リレーションシップ】')
    for r in rels:
        lines.append(f'  - {r["relationshipType"]}')
    lines.append('\n【ノードのプロパティ】')
    current_label = None
    for r in node_props:
        label = str(r['nodeLabels'])
        if label != current_label:
            lines.append(f'  {label}:')
            current_label = label
        lines.append(f'    - {r["propertyName"]} ({r["propertyTypes"]})')
    if rel_props:
        lines.append('\n【リレーションシップのプロパティ】')
        for r in rel_props:
            lines.append(f'  {r["relType"]}: {r["propertyName"]} ({r["propertyTypes"]})')
    return '\n'.join(lines)

# 確認
print(get_schema())

---
## STEP 8 — Graph RAG パイプラインの定義

### 設計のポイント

LLM に Cypher を直接生成させると、リレーション名の誤り・複数ホップのまとめなどが発生します。
そこで役割を分離しています。

```
LLM の役割    : 「何を・どのリレーションで・どの順にたどるか」を JSON で返す
コードの役割  : その JSON から Cypher 文字列を生成する
```

これにより **1ステップ = 1リレーション = 1クエリ** が保証されます。

In [ ]:
import requests, json, re

OLLAMA_URL = 'http://localhost:11434/api/generate'

# LLM には Cypher を書かせず「検索ステップの意図」だけ返させる
PLANNER_PROMPT = """あなたはグラフデータベースの検索プランナーです。
ユーザーの質問を解くために必要な「検索ステップ」を順番にリストアップしてください。

以下はグラフのスキーマです:
{schema}

各ステップは以下の形式で返してください:
[
  {{"action": "find_related", "from_label": "Person", "from_name": "田中太郎", "rel": "BELONGS_TO", "to_label": "Org", "return_as": "org"}},
  {{"action": "find_related", "from_label": "Org",    "from_name": "{{org}}",   "rel": "PART_OF",    "to_label": "Company", "return_as": "company"}},
  {{"action": "find_related", "from_label": "Person", "from_name": null,        "rel": "REPRESENTS", "to_label": "Company", "filter_by": "company", "return_as": "representative"}}
]

ルール:
- 1ステップ = 1リレーション
- from_name が前ステップの結果に依存する場合は {{変数名}} と書く
- スキーマに存在するリレーションのみ使用すること
- JSONのみ返すこと

質問: {question}"""


def call_ollama(prompt: str) -> str:
    resp = requests.post(OLLAMA_URL, json={
        'model': MODEL_NAME,
        'prompt': prompt,
        'stream': False,
        'options': {'temperature': 0.0}
    }, timeout=180)
    resp.raise_for_status()
    return resp.json()['response']


def build_cypher_from_step(step: dict, accumulated: dict) -> str:
    """
    LLM が返したステップ定義から Cypher を自動生成する。
    Cypher の文字列生成はコード側で行い、LLM の誤りを排除する。
    """
    from_label = step.get('from_label', '')
    rel        = step.get('rel', '')
    to_label   = step.get('to_label', '')
    return_as  = step.get('return_as', 'result')
    from_name  = step.get('from_name')
    filter_by  = step.get('filter_by')

    # from_name の {変数} を前クエリの結果で置換
    if from_name and str(from_name).startswith('{') and str(from_name).endswith('}'):
        key = str(from_name)[1:-1]
        values = accumulated.get(key, [])
        if values:
            names_str = ', '.join([f"'{v}'" for v in values])
            return (f"MATCH (a:{from_label})-[:{rel}]->(b:{to_label})"
                    f" WHERE a.name IN [{names_str}] RETURN b.name AS {return_as}")
        else:
            return f"MATCH (a:{from_label})-[:{rel}]->(b:{to_label}) RETURN b.name AS {return_as}"

    elif from_name:
        return (f"MATCH (a:{from_label} {{name:'{from_name}'}})"
                f"-[:{rel}]->(b:{to_label}) RETURN b.name AS {return_as}")

    elif filter_by:
        values = accumulated.get(filter_by, [])
        if values:
            names_str = ', '.join([f"'{v}'" for v in values])
            return (f"MATCH (a:{from_label})-[:{rel}]->(b:{to_label})"
                    f" WHERE b.name IN [{names_str}] RETURN a.name AS {return_as}")
        else:
            return f"MATCH (a:{from_label})-[:{rel}]->(b:{to_label}) RETURN a.name AS {return_as}"

    else:
        return f"MATCH (a:{from_label})-[:{rel}]->(b:{to_label}) RETURN b.name AS {return_as}"


def plan_cypher_queries(user_question: str) -> list:
    """
    LLM に検索ステップの意図を返させ、
    コードが Cypher を生成する。
    1ステップ = 1リレーション = 1クエリ を保証。
    """
    # ★ スキーマを Neo4j から自動取得
    schema = get_schema()
    prompt = PLANNER_PROMPT.format(schema=schema, question=user_question)

    raw = call_ollama(prompt).strip()
    raw = re.sub(r'```[a-z]*', '', raw).replace('```', '').strip()
    array_match = re.search(r'\[.*\]', raw, re.DOTALL)
    dict_match  = re.search(r'\{.*\}', raw, re.DOTALL)
    if array_match:
        steps = json.loads(array_match.group())
    elif dict_match:
        steps = json.loads(dict_match.group()).get('subqueries', [])
    else:
        raise ValueError(f'JSONが見つかりません:\n{raw}')

    # ★ ステップごとに Cypher をコードで生成
    subqueries = []
    accumulated = {}
    for step in steps:
        cypher = build_cypher_from_step(step, accumulated)
        subqueries.append({
            'cypher': cypher,
            'purpose': f"{step.get('from_label')} -[{step.get('rel')}]-> {step.get('to_label')}"
        })
        accumulated[step.get('return_as', 'result')] = []
    return subqueries


def run_graph_rag(user_question: str) -> dict:
    """
    Graph RAG パイプライン（スキーマ自動取得版）
      [アプリ層] Neo4j からスキーマを自動取得
      [アプリ層] Ollama が検索ステップの意図を返す
      [アプリ層] コードが Cypher を生成（1ステップ=1クエリ保証）
      [DB 層]   Neo4j に1件ずつ投入・実行
      [アプリ層] Ollama が結果を統合して最終回答を生成
    """
    sep = '=' * 65
    print(f'\n{sep}')
    print(f' ユーザークエリ: {user_question}')
    print(sep)

    # Step A: スキーマ取得 → LLM が検索ステップを計画 → コードが Cypher 生成
    print('\n[STEP A] スキーマ自動取得 → Ollama がステップ計画 → Cypher 生成...')
    subqueries = plan_cypher_queries(user_question)
    print(f'  → {len(subqueries)} 件の Cypher クエリを生成')
    for i, sq in enumerate(subqueries, 1):
        print(f'\n  ┌ クエリ {i}/{len(subqueries)}')
        print(f'  │ 目的   : {sq.get("purpose", "")}')
        print(f'  │ Cypher : {sq.get("cypher", "")}')
        print(f'  └' + '─' * 50)

    # Step B: Neo4j に各クエリを個別投入
    print('\n[STEP B] Neo4j（DB層）に各 Cypher クエリを1件ずつ投入...')
    print('  ↑ クエリ分解はアプリ層が実施。Neo4j は1件ずつ受け取るだけ')

    all_results = []
    accumulated = {}

    for i, sq in enumerate(subqueries, 1):
        cypher = sq.get('cypher', '')
        # 前クエリの結果で {xxx} プレースホルダを置換
        for key, values in accumulated.items():
            placeholder = '{' + key + '}'
            if placeholder in cypher:
                values_str = ', '.join([f"'{v}'" for v in values])
                cypher = cypher.replace(f"['{placeholder}']", f'[{values_str}]')
                cypher = cypher.replace(placeholder, values_str)

        print(f'\n  Neo4j クエリ {i}/{len(subqueries)}: {cypher}')
        try:
            result = run_cypher(cypher)
        except Exception as e:
            result = [{'error': str(e)}]
        print(f'  結果: {result}')

        for row in result:
            for col, val in row.items():
                if isinstance(val, dict):
                    continue
                accumulated.setdefault(col, [])
                if str(val) not in accumulated[col]:
                    accumulated[col].append(str(val))

        all_results.append({'purpose': sq.get('purpose'), 'cypher': cypher, 'result': result})

    # Step C: LLM が結果を統合して回答生成
    print('\n[STEP C] Ollama（アプリ層）が結果を統合して回答を生成中...')
    context = json.dumps(all_results, ensure_ascii=False, indent=2)
    answer = call_ollama(
        f'以下のNeo4jクエリ結果をもとに、質問に日本語で簡潔に答えてください。\n'
        f'質問: {user_question}\n\nクエリ結果:\n{context}\n\n回答:'
    ).strip()

    print(f'\n{sep}')
    print(f' 最終回答: {answer}')
    print(sep)
    print(f'\n 📊 Neo4j へのクエリ数: {len(subqueries)} 件')
    print(f'    （1つのユーザークエリが {len(subqueries)} つの Cypher クエリに展開されました）')
    return {'answer': answer, 'query_count': len(subqueries)}


print('Ollama 接続テスト:', call_ollama("'OK'とだけ答えてください")[:20])
print('✅ パイプライン定義完了')

---
## STEP 9 — デモ実行 ① : 2ホップの質問
**「田中太郎が所属する組織を管理している人は誰ですか？」**

期待: `BELONGS_TO`（1件）→ `MANAGES`（1件）= **2件**

In [ ]:
result1 = run_graph_rag('田中太郎が所属する組織を管理している人は誰ですか？')

---
## STEP 10 — デモ実行 ② : 3ホップの質問
**「田中太郎が所属する組織が属する会社の代表者は誰ですか？」**

期待: `BELONGS_TO`（1件）→ `PART_OF`（1件）→ `REPRESENTS`（1件）= **3件**

In [ ]:
result2 = run_graph_rag('田中太郎が所属する組織が属する会社の代表者は誰ですか？')

---
## STEP 11 — デモ実行 ③ : 広範囲の質問
**「開発部のメンバー全員と、各メンバーが参加しているプロジェクトを教えてください」**

期待: `BELONGS_TO`（1件）→ `WORKS_ON`（1件）= **2件**

In [ ]:
result3 = run_graph_rag('開発部のメンバー全員と、各メンバーが参加しているプロジェクトを教えてください')

---
## STEP 12 — 結果の比較グラフ

ホップ数 = クエリ数になっていることを確認します。

In [ ]:
import os; os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib'
import matplotlib.font_manager as fm, matplotlib.pyplot as plt
fm.fontManager.addfont('/usr/share/fonts/opentype/ipafont-gothic/ipagp.ttf')
plt.rcParams['font.family'] = 'IPAPGothic'

questions = ['Q1\n2ホップ\n（組織の管理者は？）',
             'Q2\n3ホップ\n（会社の代表者は？）',
             'Q3\n広範囲\n（メンバーとPJは？）']
counts    = [result1['query_count'], result2['query_count'], result3['query_count']]
expected  = [2, 3, 2]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.bar(questions, counts, color=['#7F77DD', '#1D9E75', '#D85A30'], width=0.5)
ax.bar_label(bars, labels=[f'{v}件' for v in counts], padding=5, fontsize=14)

# 期待値をマーカーで表示
for i, e in enumerate(expected):
    ax.plot(i, e, 'x', color='black', markersize=12, markeredgewidth=2,
            label='期待値' if i == 0 else '')

ax.set_ylabel('Neo4j への Cypher クエリ数', fontsize=12)
ax.set_title('1つのユーザークエリ → Neo4j に投げられる Cypher クエリ数\n'
             '（スキーマ自動取得版 / ホップ数 = クエリ数を保証）', fontsize=12)
ax.set_ylim(0, max(counts) + 2)
ax.axhline(y=1, color='gray', linestyle='--', alpha=0.5, label='クエリ数=1（単純なDB検索）')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('graph_rag_auto_schema_result.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n【まとめ】')
print('・スキーマは Neo4j から自動取得（ハードコード不要）')
print('・LLM は検索ステップの意図のみを返す')
print('・Cypher の生成はコード側で実施（1ステップ=1リレーション=1クエリ保証）')
print('・Neo4j は受け取った Cypher を実行するだけ  ← DB 層')

---
## STEP 13 — スキーマ確認とアーキテクチャまとめ

In [ ]:
print('=' * 60)
print('Neo4j から自動取得したスキーマ情報')
print('（これが実行時にプロンプトへ動的に埋め込まれます）')
print('=' * 60)
print(get_schema())
print()
print('=' * 60)
print('アーキテクチャ比較')
print()
print('  固定版')
print('    LLM → Cypher文字列を直接生成（リレーション誤りが発生しやすい）')
print('    スキーマ変更時 → プロンプトの書き直しが必要')
print()
print('  自動取得版（本ノートブック）')
print('    LLM → 検索ステップの意図（from/rel/to）のみ返す')
print('    コード → 意図から Cypher を生成（1ステップ=1クエリ保証）')
print('    スキーマ変更時 → 自動対応（コード変更不要）')
print('=' * 60)